#### **Middleware**

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

- Tracking agent behavior with logging, analytics, and debugging.
- Transforming prompts, tool selection, and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails, and PII detection.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv() ##Loads the environment variables from a .env file into the system's environment variables

os.environ["ABACUS_API_KEY"]=os.getenv("ABACUS_API_KEY")

In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o",
    api_key=os.environ["ABACUS_API_KEY"],
    base_url="https://routellm.abacus.ai/v1",
)

#### **Summarization Middleware**

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

In [4]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### Messagebased summarization
agent=create_agent(
    model=llm,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [5]:
### Run with thread id
config={"configurable":{"thread_id":"test-1"}}

In [6]:
# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='9a34dab9-feb5-4441-83ce-0f65bfb36e50'), AIMessage(content='2 + 2 equals 4.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': None, 'prompt_tokens': None, 'total_tokens': None, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'input_tokens': 14, 'output_tokens': 8, 'raw_input_tokens': 14}, 'model_provider': 'openai', 'model_name': 'gpt-4o', 'system_fingerprint': None, 'id': 'chatcmpl-3b528873b5cd46e1a51d872590835a9b', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a13cf3-fdb3-7f81-9c21-b0ead8c7d91e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 0, 'output_tokens': 0, 'total_tokens': 0, 'input_token_details': {}, 'output_token_details': {}})]}
Messages: 2
Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='9a34dab9-feb5-4441-8

**Token Size**

In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent=create_agent(
    model=llm,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("tokens",550),
            keep=("tokens",200),
        ),
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token


In [9]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~137 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='97d1e724-7fbd-40b5-b723-97862fbc7f58'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': None, 'prompt_tokens': None, 'total_tokens': None, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'input_tokens': 53, 'output_tokens': 15, 'raw_input_tokens': 53}, 'model_provider': 'openai', 'model_name': 'gpt-4o', 'system_fingerprint': None, 'id': 'chatcmpl-f008868e82b045bf865093c655b98bb9', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a13cf5-a2cb-77c0-937c-082e1bd39ce8-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'call_wLSXJ6MkymmMBKERFsGTi4w5', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 0, 'output_tokens': 0, 'total_tokens': 0, 'input_token_details': {}, 'output_token_details': {}}), ToolMessage(content='H

**Fraction**

In [11]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
agent = create_agent(
    model=llm,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
            keep=("fraction", 0.002),     # 0.2% = ~256 tokens
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  # gpt-4o-mini context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

Paris: ~64 tokens (0.0500%), 4 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='8c20ea03-c9f6-43e9-b3aa-1cec50981e9e'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': None, 'prompt_tokens': None, 'total_tokens': None, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'input_tokens': 44, 'output_tokens': 15, 'raw_input_tokens': 44}, 'model_provider': 'openai', 'model_name': 'gpt-4o', 'system_fingerprint': None, 'id': 'chatcmpl-f8765b83105b454cb1ec217c80a54619', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a13cf6-eebe-79e2-b6dc-cf4c7495d6b5-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'call_BsffHzZNKRAhttzyCmy10ikP', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 0, 'output_tokens': 0, 'total_tokens': 0, 'input_token_details': {}, 'output_token_details': {}}), ToolMessage(content='H

#### **Human In the Loop MiddleWare**

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

- High-stakes operations requiring human approval (e.g. database writes, financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [12]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [15]:
agent=create_agent(
    model=llm,
    tools=[read_email_tool,send_email_tool], ##Can pass multiple tools to the agent
    checkpointer=InMemorySaver(), ## Checkpointer to save the state of the agent
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False, ## No human intervention for reading emails

            }
        )
    ]
)

In [16]:
config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [17]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='dc7ac6cf-151f-4cc2-9a94-83e922840f7c'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': None, 'prompt_tokens': None, 'total_tokens': None, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'input_tokens': 97, 'output_tokens': 28, 'raw_input_tokens': 97}, 'model_provider': 'openai', 'model_name': 'gpt-4o', 'system_fingerprint': None, 'id': 'chatcmpl-db9a1ba428fd465ba25f97614914df7f', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a13cf9-2b6b-7e32-83e4-9308781031a8-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@test.com', 'subject': 'Hello', 'body': 'How are you?'}, 'id': 'call_waQ3h2YexhSh1QZRURiUnN78', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 0, 'output_tokens': 0, 'to

In [18]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: The email has been sent to john@test.com with the subject "Hello" and the body "How are you?".


In [19]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='dc7ac6cf-151f-4cc2-9a94-83e922840f7c'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': None, 'prompt_tokens': None, 'total_tokens': None, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'input_tokens': 97, 'output_tokens': 28, 'raw_input_tokens': 97}, 'model_provider': 'openai', 'model_name': 'gpt-4o', 'system_fingerprint': None, 'id': 'chatcmpl-db9a1ba428fd465ba25f97614914df7f', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a13cf9-2b6b-7e32-83e4-9308781031a8-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@test.com', 'subject': 'Hello', 'body': 'How are you?'}, 'id': 'call_waQ3h2YexhSh1QZRURiUnN78', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 0, 'output_tokens': 0, 'to

**Reject**

In [21]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model=llm,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [22]:
config = {"configurable": {"thread_id": "test-reject"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config)

In [23]:
# Step 2: Reject
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: It seems that the email send request was not processed. If you would like to attempt sending the email again or if you have any specific requirements, please let me know!


In [24]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='622b5d4c-0909-4d94-96c6-d01d7835643e'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': None, 'prompt_tokens': None, 'total_tokens': None, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'input_tokens': 97, 'output_tokens': 28, 'raw_input_tokens': 97}, 'model_provider': 'openai', 'model_name': 'gpt-4o', 'system_fingerprint': None, 'id': 'chatcmpl-f0e0b8f47b424f23b289d20d60e2e47f', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a13cfa-b1da-7bd3-b890-69f11f7bfda0-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@test.com', 'subject': 'Hello', 'body': 'How are you?'}, 'id': 'call_8Y2sbqmEETVCpUsDJyQLZGTa', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 0, 'output_tokens': 0, 'to

**Editing**

In [26]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model=llm,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [27]:
config = {"configurable": {"thread_id": "test-edit"}}

# Step 1: Request (with wrong info)
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'")]},
    config=config
)

In [28]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='5db132d7-977c-4379-b1d6-6c57450f1a0e'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': None, 'prompt_tokens': None, 'total_tokens': None, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'input_tokens': 95, 'output_tokens': 25, 'raw_input_tokens': 95}, 'model_provider': 'openai', 'model_name': 'gpt-4o', 'system_fingerprint': None, 'id': 'chatcmpl-c3b2edd1bfc14a30bf4259841bc5cbcb', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a13cfb-aff7-7da1-90d5-87211109d493-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'wrong@email.com', 'subject': 'Test', 'body': 'Hello'}, 'id': 'call_div4EJoE4Mb7JBOOKMsV8RvX', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 0, 'output_tokens': 0, 'total_tokens':

In [29]:
# Step 2: Edit and approve
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                   # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
    
    print(f"✏️ Result: {result['messages'][-1].content}")

⏸️ Paused! Editing...
✏️ Result: It seems there was a correction before the email was sent, and it was directed to a different address with different content. If you need any further assistance, feel free to ask!


In [30]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='5db132d7-977c-4379-b1d6-6c57450f1a0e'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': None, 'prompt_tokens': None, 'total_tokens': None, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'input_tokens': 95, 'output_tokens': 25, 'raw_input_tokens': 95}, 'model_provider': 'openai', 'model_name': 'gpt-4o', 'system_fingerprint': None, 'id': 'chatcmpl-c3b2edd1bfc14a30bf4259841bc5cbcb', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a13cfb-aff7-7da1-90d5-87211109d493-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'wrong@email.com', 'subject': 'Test', 'body': 'Hello'}, 'id': 'call_div4EJoE4Mb7JBOOKMsV8RvX', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 0, 'output_tokens': 0, 'total_tokens':